# Corrupted Image Dataset Generation

Downloads commercial-compatible Wikimedia Commons images, splits raw sources before corruption, creates corrupted 384x384 samples, and writes strict 4-column labels.

In [27]:
import csv
from collections import Counter
import json
import random
import re
import time
from pathlib import Path
from urllib.parse import urlparse
import os
import hashlib

import numpy as np
import requests
from IPython.display import display
from PIL import Image, ImageOps, UnidentifiedImageError
from tqdm.auto import tqdm

In [30]:
SEED = 42
TARGET_SAMPLES = 40_000
RAW_IMAGE_TARGET = 10_000
COMMONS_SEARCH_PAGES = 100
COMMONS_SEARCH_PAGE_SIZE = 50
DOWNLOAD_RETRIES = 4
DOWNLOAD_SLEEP_SECONDS = 0.4
IMAGE_SIZE = 384
MAX_SOURCE_PIXELS = 15_000_000

BRIGHTNESS_RANGE = (-0.5, 0.5)
CONTRAST_RANGE = (0.4, 2.2)
SATURATION_RANGE = (0.4, 2.2)
SPLIT_FRACTIONS = {"train": 0.8, "val": 0.1, "test": 0.1}
LOW_DISTORTION_FRACTION = 0.25
ORIGINAL_FRACTION = 0.10
LOW_BRIGHTNESS_RANGE = (-0.08, 0.08)
LOW_CONTRAST_RANGE = (0.85, 1.15)
LOW_SATURATION_RANGE = (0.85, 1.15)
ORIGINAL_TARGET = {"brightness": 0.0, "contrast": 1.0, "saturation": 1.0}
DISTORTION_LEVELS = ("none", "small", "high")
LABEL_FIELDS = ["image_id", "brightness", "contrast", "saturation", "distortion_level"]

AUGMENT_FLIP_H = True
AUGMENT_FLIP_V = True
AUGMENT_ROTATE_90 = True

COMMONS_API = "https://commons.wikimedia.org/w/api.php"
USER_AGENT = "practice-image-enhancer-dataset/1.0 (local notebook; https://commons.wikimedia.org/)"
SEARCH_QUERIES = [
    "nature landscape",
    "city street",
    "food market",
    "people portrait",
    "vehicles transport",
    "animals wildlife",
    "architecture building",
    "objects still life",
    "mountains lake",
    "forest path",
    "beach ocean",
    "flowers garden",
    "interior room",
    "technology device",
    "sports event",
    "public domain photograph",
]

ROOT = Path.cwd().parent.absolute()

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
SPLIT_DIRS = {split: PROCESSED_DIR / split for split in SPLIT_FRACTIONS}
LABELS_CSV = PROCESSED_DIR / "labels.csv"
CANDIDATES_CACHE = DATA_DIR / "candidates.json"
DOWNLOADED_HASHES = DATA_DIR / "downloaded_hashes.txt"

for directory in (RAW_DIR, *SPLIT_DIRS.values()):
    directory.mkdir(parents=True, exist_ok=True)

rng = random.Random(SEED)
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

print(f"Root: {ROOT}")
print(f"Data: {DATA_DIR}")
print(f"Output: {PROCESSED_DIR}")

Root: c:\Users\Марсель\Documents\practice\ml
Data: c:\Users\Марсель\Documents\practice\ml\data
Output: c:\Users\Марсель\Documents\practice\ml\data\processed


In [31]:
def clean_html(value):
    if not value:
        return ""
    return re.sub(r"<[^>]+>", "", str(value)).strip()


def ext_value(extmetadata, key):
    return clean_html(extmetadata.get(key, {}).get("value", ""))


def license_is_commercial_compatible(short_name, license_url):
    text = f"{short_name} {license_url}".lower()
    deny_patterns = [
        "noncommercial",
        "non-commercial",
        "no derivatives",
        "noderivatives",
        "fair use",
        "copyrighted free use",
    ]
    if any(pattern in text for pattern in deny_patterns):
        return False
    if re.search(r"(^|[^a-z])nc($|[^a-z])", text) or re.search(r"(^|[^a-z])nd($|[^a-z])", text):
        return False
    allow_patterns = ["public domain", "cc0", "cc by", "cc-by"]
    return any(pattern in text for pattern in allow_patterns)


def safe_stem(value):
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._-")
    return stem[:120] or "image"


def suffix_from_url(url, mime):
    suffix = Path(urlparse(url).path).suffix.lower()
    if suffix in {".jpg", ".jpeg", ".png", ".webp"}:
        return suffix
    return {"image/jpeg": ".jpg", "image/png": ".png", "image/webp": ".webp"}.get(mime, ".jpg")


def commons_request(params, retries=DOWNLOAD_RETRIES):
    last_error = None
    for attempt in range(retries):
        try:
            response = session.get(COMMONS_API, params=params, timeout=30)
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as error:
            last_error = error
            time.sleep(DOWNLOAD_SLEEP_SECONDS * (2 ** attempt))
    raise RuntimeError(f"Commons API request failed after {retries} attempts: {last_error}")


def commons_search(query, search_counts, per_page=COMMONS_SEARCH_PAGE_SIZE, max_pages=COMMONS_SEARCH_PAGES):
    candidates = []
    continuation = {}
    for _ in range(max_pages):
        params = {
            "action": "query",
            "format": "json",
            "generator": "search",
            "gsrsearch": query,
            "gsrnamespace": 6,
            "gsrlimit": per_page,
            "prop": "imageinfo",
            "iiprop": "url|size|mime|extmetadata",
        }
        params.update(continuation)
        payload = commons_request(params)
        pages = payload.get("query", {}).get("pages", {})

        for page in pages.values():
            search_counts["seen"] += 1
            imageinfo = page.get("imageinfo") or []
            if not imageinfo:
                search_counts["missing_imageinfo"] += 1
                continue
            info = imageinfo[0]
            mime = info.get("mime", "")
            width = int(info.get("width") or 0)
            height = int(info.get("height") or 0)
            pixels = width * height
            if mime not in {"image/jpeg", "image/png", "image/webp"}:
                search_counts["unsupported_mime"] += 1
                continue
            if min(width, height) < IMAGE_SIZE:
                search_counts["too_small"] += 1
                continue
            if pixels > MAX_SOURCE_PIXELS:
                search_counts["too_large"] += 1
                continue

            extmetadata = info.get("extmetadata", {})
            license_name = ext_value(extmetadata, "LicenseShortName")
            license_url = ext_value(extmetadata, "LicenseUrl")
            if not license_is_commercial_compatible(license_name, license_url):
                search_counts["license_rejected"] += 1
                continue

            url = info.get("url")
            if not url:
                search_counts["missing_url"] += 1
                continue
            search_counts["candidate_ok"] += 1

            candidates.append({
                "pageid": page.get("pageid"),
                "title": page.get("title", ""),
                "url": url,
                "mime": mime,
                "width": width,
                "height": height,
                "pixels": pixels,
            })

        if "continue" not in payload:
            break
        continuation = payload["continue"]
        time.sleep(0.1)
    return candidates

In [35]:
def image_loads(path):
    """Check if an image file is valid and meets size constraints."""
    try:
        with Image.open(path) as image:
            width, height = image.size
            if min(width, height) < IMAGE_SIZE:
                return False, "too_small"
            if width * height > MAX_SOURCE_PIXELS:
                return False, "too_large"
            image.load()
            return True, "ok"
    except (OSError, UnidentifiedImageError):
        return False, "bad_image"


def file_sha256(path):
    """Return the SHA‑256 hex digest of a file."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def load_candidates_cache():
    """Load cached candidates from JSON."""
    if CANDIDATES_CACHE.exists():
        with CANDIDATES_CACHE.open("r") as f:
            return json.load(f)
    return None


def save_candidates_cache(candidates):
    """Save candidates to JSON cache."""
    with CANDIDATES_CACHE.open("w") as f:
        json.dump(candidates, f, indent=2)


def load_downloaded_hashes():
    """Load already downloaded image hashes."""
    if DOWNLOADED_HASHES.exists():
        with DOWNLOADED_HASHES.open("r") as f:
            return set(line.strip() for line in f if line.strip())
    return set()


def save_downloaded_hashes(hashes_set):
    """Persist the set of downloaded hashes."""
    with DOWNLOADED_HASHES.open("w") as f:
        for h in sorted(hashes_set):
            f.write(h + "\n")

def download_candidate(candidate, skip_counts, downloaded_hashes):
    """Download a single candidate, save it using its SHA‑256 hash, return path or None."""
    suffix = suffix_from_url(candidate["url"], candidate["mime"])
    last_error = None
    tmp_path = None

    for attempt in range(DOWNLOAD_RETRIES):
        try:
            resp = session.get(candidate["url"], stream=True, timeout=90)
            if resp.status_code == 429:
                retry_after = resp.headers.get("Retry-After")
                if retry_after:
                    wait = int(retry_after)
                else:
                    wait = DOWNLOAD_SLEEP_SECONDS * (2 ** attempt)
                print(f"Rate limited, waiting {wait} seconds...")
                time.sleep(wait)
                skip_counts["rate_limited"] += 1
                continue

            resp.raise_for_status()

            tmp_path = RAW_DIR / f".tmp_{os.getpid()}_{time.time_ns()}{suffix}"
            with tmp_path.open("wb") as f:
                for chunk in resp.iter_content(chunk_size=256*1024):
                    if chunk:
                        f.write(chunk)

            valid, reason = image_loads(tmp_path)
            if not valid:
                tmp_path.unlink(missing_ok=True)
                skip_counts[reason] += 1
                return None

            h = file_sha256(tmp_path)
            final_path = RAW_DIR / f"{h}{suffix}"
            if final_path.exists():
                tmp_path.unlink(missing_ok=True)
                skip_counts["duplicate_hash"] += 1
                downloaded_hashes.add(h)
                return None

            tmp_path.rename(final_path)
            downloaded_hashes.add(h)
            time.sleep(DOWNLOAD_SLEEP_SECONDS)
            return final_path

        except requests.RequestException as error:
            last_error = error
            if tmp_path is not None and tmp_path.exists():
                tmp_path.unlink(missing_ok=True)
            wait = DOWNLOAD_SLEEP_SECONDS * (2 ** attempt)
            print(f"Request failed (attempt {attempt+1}), waiting {wait}s: {error}")
            time.sleep(wait)

    skip_counts["request_failed"] += 1
    skip_counts[f"last_error:{type(last_error).__name__}"] += 1
    return None

def load_raw_sources_from_hashes(downloaded_hashes):
    """Scan RAW_DIR for existing hash‑named files and return a list of source dicts."""
    sources = []
    for raw_path in tqdm(sorted(RAW_DIR.iterdir()), desc="Loading images"):
        if not raw_path.is_file():
            continue
        valid, _ = image_loads(raw_path)
        if not valid:
            continue

        h = raw_path.stem
        if h not in downloaded_hashes:
            downloaded_hashes.add(h)
        sources.append({"raw_path": str(raw_path.relative_to(ROOT)), "hash": h})
    return sources

print(f"Checking caches in {DOWNLOADED_HASHES}")
downloaded_hashes = load_downloaded_hashes()
print(f"Loading cached images from {RAW_DIR}")
sources = load_raw_sources_from_hashes(downloaded_hashes)
print(f"Already downloaded images: {len(sources)}")

cached = load_candidates_cache()
if cached:
    print(f"Loaded {len(cached)} candidates from cache.")
    all_candidates = cached
else:
    all_candidates = []
    seen_urls = set()
    search_counts = Counter()
    for query in SEARCH_QUERIES:
        print(f"Searching Commons: {query}")
        for candidate in commons_search(query, search_counts):
            if candidate["url"] in seen_urls:
                continue
            seen_urls.add(candidate["url"])
            all_candidates.append(candidate)
        if len(all_candidates) >= RAW_IMAGE_TARGET * 5:
            break
    rng.shuffle(all_candidates)
    save_candidates_cache(all_candidates)
    print(f"Candidates cached: {len(all_candidates)}")

skip_counts = Counter()
for candidate in tqdm(all_candidates, desc="Downloading new images", total=min(RAW_IMAGE_TARGET, len(all_candidates))):
    if len(sources) >= RAW_IMAGE_TARGET:
        break
    path = download_candidate(candidate, skip_counts, downloaded_hashes)
    if path is None:
        continue

    h = path.stem
    sources.append({"raw_path": str(path.relative_to(ROOT)), "hash": h})

    seen = set()
    sources = [s for s in sources if s["raw_path"] not in seen and not seen.add(s["raw_path"])]

save_downloaded_hashes(downloaded_hashes)
print(f"Total usable images: {len(sources)}")
print(f"Download skip counts: {dict(skip_counts)}")

split_sources = {split: [] for split in SPLIT_FRACTIONS}
rng.shuffle(sources)
train_count = int(len(sources) * SPLIT_FRACTIONS["train"])
val_count = int(len(sources) * SPLIT_FRACTIONS["val"])
split_sources["train"] = sources[:train_count]
split_sources["val"] = sources[train_count:train_count + val_count]
split_sources["test"] = sources[train_count + val_count:]

for split, srcs in split_sources.items():
    if not srcs:
        raise RuntimeError(f"No raw sources assigned to {split} split.")
    print(f"{split}: {len(srcs)} raw images")

Checking caches in c:\Users\Марсель\Documents\practice\ml\data\downloaded_hashes.txt
Loading cached images from c:\Users\Марсель\Documents\practice\ml\data\raw


Loading images:   0%|          | 0/10000 [00:00<?, ?it/s]

Already downloaded images: 10000
Loaded 48574 candidates from cache.


Total usable images: 10000
Download skip counts: {}
train: 8000 raw images
val: 1000 raw images
test: 1000 raw images


In [36]:
def clean_preview(raw_path):
    """Load, orient, resize to IMAGE_SIZE centre crop."""
    with Image.open(raw_path) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        image.load()
        return ImageOps.fit(image, (IMAGE_SIZE, IMAGE_SIZE),
                            method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))

def apply_geometric_augmentation(image, flip_h, flip_v, rotate_k):
    """Apply flips and 90‑degree rotations to a PIL image."""
    if flip_h:
        image = image.transpose(Image.FLIP_LEFT_RIGHT)
    if flip_v:
        image = image.transpose(Image.FLIP_TOP_BOTTOM)
    if rotate_k:
        image = image.rotate(90 * rotate_k, expand=True)
    return image

def make_corrupted_image(clean_image, brightness, contrast, saturation):
    """Pixel‑level brightness/contrast/saturation distortion."""
    clean = np.asarray(clean_image, dtype=np.float32)
    gray = clean[..., 0] * 0.2126 + clean[..., 1] * 0.7152 + clean[..., 2] * 0.0722
    before_saturation = gray[..., None] + (clean - gray[..., None]) / saturation
    corrupted = ((before_saturation - 128.0 - brightness * 255.0) / contrast) + 128.0
    corrupted = np.clip(corrupted, 0, 255).astype(np.uint8)
    return Image.fromarray(corrupted, mode="RGB")

def split_sample_counts(total):
    train_count = int(total * SPLIT_FRACTIONS["train"])
    val_count = int(total * SPLIT_FRACTIONS["val"])
    return {"train": train_count, "val": val_count, "test": total - train_count - val_count}

def sample_target(split_index, split_total):
    original_count = int(split_total * ORIGINAL_FRACTION)
    low_distortion_count = int(split_total * LOW_DISTORTION_FRACTION)
    if split_index < original_count:
        return dict(ORIGINAL_TARGET), "none"
    if split_index < original_count + low_distortion_count:
        return {
            "brightness": rng.uniform(*LOW_BRIGHTNESS_RANGE),
            "contrast": rng.uniform(*LOW_CONTRAST_RANGE),
            "saturation": rng.uniform(*LOW_SATURATION_RANGE),
        }, "small"
    return {
        "brightness": rng.uniform(*BRIGHTNESS_RANGE),
        "contrast": rng.uniform(*CONTRAST_RANGE),
        "saturation": rng.uniform(*SATURATION_RANGE),
    }, "high"

def image_path_for_id(image_id):
    split = image_id.split("_", 1)[0]
    return SPLIT_DIRS[split] / f"{image_id}.jpg"

rows = []
generation_skips = Counter()
split_counts = split_sample_counts(TARGET_SAMPLES)
global_index = 0
progress = tqdm(total=TARGET_SAMPLES, desc="Generating corrupted samples")
for split, split_total in split_counts.items():
    split_index = 0
    while split_index < split_total:
        if not split_sources[split]:
            raise RuntimeError(f"No usable source images remain in {split} split.")
        source = rng.choice(split_sources[split])
        raw_path = ROOT / source["raw_path"]
        image_id = f"{split}_{split_index:06d}"
        target, distortion_level = sample_target(split_index, split_total)

        try:
            clean = clean_preview(raw_path)
        except (OSError, UnidentifiedImageError) as error:
            generation_skips[type(error).__name__] += 1
            split_sources[split].remove(source)
            raw_path.unlink(missing_ok=True)
            continue

        corrupted = make_corrupted_image(clean, target["brightness"],
                                         target["contrast"], target["saturation"])

        flip_h = AUGMENT_FLIP_H and rng.choice([True, False])
        flip_v = AUGMENT_FLIP_V and rng.choice([True, False])
        rotate_k = 0
        if AUGMENT_ROTATE_90:
            rotate_k = rng.randint(0, 3)   # 0, 1, 2, 3 -> 0, 90, 180, 270
        corrupted = apply_geometric_augmentation(corrupted, flip_h, flip_v, rotate_k)

        out_path = SPLIT_DIRS[split] / f"{image_id}.jpg"
        corrupted.save(out_path, format="JPEG", quality=92, optimize=True)

        rows.append({
            "image_id": image_id,
            "brightness": f"{target['brightness']:.8f}",
            "contrast": f"{target['contrast']:.8f}",
            "saturation": f"{target['saturation']:.8f}",
            "distortion_level": distortion_level,
        })
        split_index += 1
        global_index += 1
        progress.update(1)
progress.close()

with LABELS_CSV.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=LABEL_FIELDS, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)

print(f"Generation skipped bad sources: {dict(generation_skips)}")
print(f"Saved images: {PROCESSED_DIR / '<split>'}")
print(f"Saved labels: {LABELS_CSV}")

Generating corrupted samples:   0%|          | 0/40000 [00:00<?, ?it/s]

c:\Users\Марсель\AppData\Local\Programs\Python\Python314\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))
c:\Users\Марсель\AppData\Local\Programs\Python\Python314\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))
c:\Users\Марсель\AppData\Local\Programs\Python\Python314\Lib\site-packages\PIL\Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Generation skipped bad sources: {}
Saved images: c:\Users\Марсель\Documents\practice\ml\data\processed\<split>
Saved labels: c:\Users\Марсель\Documents\practice\ml\data\processed\labels.csv
